In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ananyasharma05/olist-marketplace-analytics-data/customer_features.csv
/kaggle/input/datasets/ananyasharma05/olist-marketplace-analytics-data/cohort_retention_matrix.csv
/kaggle/input/datasets/ananyasharma05/olist-marketplace-analytics-data/olist_master_clean.csv


# CUPED Variance Reduction

## Objective

Previous notebooks demonstrated that delivery performance influences customer satisfaction.

However, experimentation efficiency can be improved by reducing variance in outcome metrics.

This notebook applies CUPED (Controlled Experiments Using Pre-Experiment Data) to reduce noise and increase experiment sensitivity.

Business Question:

Can we detect the same treatment effect using fewer observations?

# 1. Imports

In [4]:
from scipy import stats

import seaborn as sns
import matplotlib.pyplot as plt

# 2. Load Data

In [5]:
master = pd.read_csv("/kaggle/input/datasets/ananyasharma05/olist-marketplace-analytics-data/olist_master_clean.csv")

master.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,...,on_time_delivery,delivery_speed_group,negative_handoff_flag,customer_order_number,first_purchase_date,days_since_first_purchase,repeat_customer_flag,cohort_month,order_month_period,cohort_index
0,e22acc9c116caa3f2b7121bbb380d08e,1,372645c7439f9661fbbacfd129aa92ec,da8622b14eb17ae2831f4ac5b9dab84a,2018-05-15 11:11:18,129.90,12.00,fadbb3709178fc513abc1b2670aa1ad2,delivered,2018-05-10 10:56:27,...,True,fast,0,1,2018-05-10 10:56:27,0,0,2018-05,2018-05,1
1,3594e05a005ac4d06a72673270ef9ec9,1,5099f7000472b634fea8304448d20825,138dbe45fc62f1e244378131a6801526,2018-05-11 17:56:33,18.90,8.29,4cb282e167ae9234755102258dd52ee8,delivered,2018-05-07 11:11:27,...,True,fast,0,1,2018-05-07 11:11:27,0,0,2018-05,2018-05,1
2,b33ec3b699337181488304f362a6b734,1,64b488de448a5324c4134ea39c28a34b,3d871de0142ce09b7081e2b9d1733cb1,2017-03-15 21:05:03,69.00,17.22,9b3932a6253894a02c1df9d19004239f,delivered,2017-03-10 21:05:03,...,True,slow,0,1,2017-03-10 21:05:03,0,0,2017-03,2017-03,1
3,41272756ecddd9a9ed0180413cc22fb6,1,2345a354a6f2033609bbf62bf5be9ef6,ef506c96320abeedfb894c34db06f478,2017-10-18 21:49:17,25.99,17.63,914991f0c02ef0843c0e7010c819d642,delivered,2017-10-12 20:29:41,...,True,slow,0,1,2017-10-12 20:29:41,0,0,2017-10,2017-10,1
4,d957021f1127559cd947b62533f484f7,1,c72e18b3fe2739b8d24ebf3102450f37,70a12e78e608ac31179aea7f8422044b,2017-11-22 20:06:52,180.00,16.89,47227568b10f5f58a524a75507e6992c,delivered,2017-11-14 19:45:42,...,True,slow,0,1,2017-11-14 19:45:42,0,0,2017-11,2017-11,1


# 3. Create Groups

In [6]:
control = master[master["delivery_speed_group"] == "slow"].copy()

treatment = master[master["delivery_speed_group"] == "fast"].copy()

In [7]:
control.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,...,on_time_delivery,delivery_speed_group,negative_handoff_flag,customer_order_number,first_purchase_date,days_since_first_purchase,repeat_customer_flag,cohort_month,order_month_period,cohort_index
2,b33ec3b699337181488304f362a6b734,1,64b488de448a5324c4134ea39c28a34b,3d871de0142ce09b7081e2b9d1733cb1,2017-03-15 21:05:03,69.00,17.22,9b3932a6253894a02c1df9d19004239f,delivered,2017-03-10 21:05:03,...,True,slow,0,1,2017-03-10 21:05:03,0,0,2017-03,2017-03,1
3,41272756ecddd9a9ed0180413cc22fb6,1,2345a354a6f2033609bbf62bf5be9ef6,ef506c96320abeedfb894c34db06f478,2017-10-18 21:49:17,25.99,17.63,914991f0c02ef0843c0e7010c819d642,delivered,2017-10-12 20:29:41,...,True,slow,0,1,2017-10-12 20:29:41,0,0,2017-10,2017-10,1
4,d957021f1127559cd947b62533f484f7,1,c72e18b3fe2739b8d24ebf3102450f37,70a12e78e608ac31179aea7f8422044b,2017-11-22 20:06:52,180.00,16.89,47227568b10f5f58a524a75507e6992c,delivered,2017-11-14 19:45:42,...,True,slow,0,1,2017-11-14 19:45:42,0,0,2017-11,2017-11,1
7,44e608f2db00c74a1fe329de44416a4e,1,62984ea1bba7fcea1f5b57084d3bf885,218d46b86c1881d022bce9c68a7d4b15,2018-03-06 11:30:57,191.00,18.59,a81ebb9b32f102298c0c89635b4b3154,delivered,2018-02-28 11:15:41,...,True,slow,0,1,2018-02-28 11:15:41,0,0,2018-02,2018-02,1
8,44e608f2db00c74a1fe329de44416a4e,2,58727e154e8e85d84052cd22b0136c84,218d46b86c1881d022bce9c68a7d4b15,2018-03-06 11:30:57,191.00,18.59,a81ebb9b32f102298c0c89635b4b3154,delivered,2018-02-28 11:15:41,...,True,slow,0,2,2018-02-28 11:15:41,0,0,2018-02,2018-02,1


# 4.  Choose CUPED Covariate
We need a variable:Known before outcome occursand correlated with:review_score

For this project use:delivery_days  because we already found:
Spearman = -0.221

In [16]:
master[["review_score", "delivery_days"]].corr(method="spearman")

,review_score,delivery_days
review_score,1.000000,-0.221332
delivery_days,-0.221332,1.000000


# 5 Calculate Theta
Formula  θ= Cov(Y,X)/Var(X)

where:
* Y = review_score
* X = delivery_days


In [8]:
valid = master[["delivery_days", "review_score"]].dropna()

X = valid["delivery_days"]

Y = valid["review_score"]

theta = np.cov(Y,X)[0,1] / np.var(X)

print(theta)

-0.04385601317303507


Theta tells us: How much outcome variation is explained by the covariate.

# 6 Create CUPED Metric

Formula: Ycuped =Y−θ(X−X^)

In [9]:
valid["review_score_cuped"] = (valid["review_score"]-theta*
    (valid["delivery_days"]-valid["delivery_days"].mean() ))

# 7. Compare Variance

In [17]:
original_var = (valid["review_score"].var())
cuped_var = (valid["review_score_cuped"].var())
reduction_pct = ((original_var - cuped_var)/original_var*100)

print("original variance", original_var)
print("cuped variance", cuped_var)

print(f"Variance Reduction: {reduction_pct:.2f}%")

original variance 1.815047055471139
cuped variance 1.6464776311586204
Variance Reduction: 9.29%


# 8. Recalculate Treatment Effect

In [13]:

valid["group"] = np.where(master.loc[valid.index,"delivery_speed_group"] == "fast",
    "treatment",
    "control")
cuped_summary = (valid.groupby("group")["review_score_cuped"].mean())

cuped_summary

group
control      4.127852
treatment    4.035822
Name: review_score_cuped, dtype: float64

In [18]:
master["delivery_speed_group"].value_counts()

delivery_speed_group
fast    55098
slow    55098
Name: count, dtype: int64

In [19]:
cuped_lift = (cuped_summary["treatment"]-cuped_summary["control"])

print(f"CUPED Lift: {cuped_lift:.4f}")

CUPED Lift: -0.0920


# 9.Effective Sample Size Gain
Formula

Gain= 1/1−R^2


In [15]:
r = valid[ ["review_score","delivery_days"]].corr().iloc[0,1]

r2 = r**2

gain = (1/(1-r2))

print(f"Effective Sample Size Gain: {gain:.3f}x")

Effective Sample Size Gain: 1.102x


### Important Limitation

The covariate selected for this demonstration was delivery_days because it is strongly correlated with review_score.

However, delivery_days occurs after the customer experiences the delivery process and is therefore not a true pre-experiment covariate.

As a result, CUPED partially removes the treatment effect itself rather than only removing unrelated noise.

This example demonstrates the mechanics of CUPED and variance reduction, but a production implementation should use variables known before treatment assignment, such as:

- freight_value
- product_weight_g
- product dimensions
- historical customer metrics

Using true pre-treatment covariates preserves treatment effects while reducing variance.

In [20]:
master[[ "review_score", "freight_value"]].corr(method="spearman")

,review_score,freight_value
review_score,1.000000,-0.045723
freight_value,-0.045723,1.000000


In [21]:
master[["review_score","total_payment_value"]].corr(method="spearman")

,review_score,total_payment_value
review_score,1.000000,-0.077164
total_payment_value,-0.077164,1.000000


In [22]:
master[["review_score","product_weight_g"]].corr(method="spearman")

,review_score,product_weight_g
review_score,1.000000,-0.016939
product_weight_g,-0.016939,1.000000


### CUPED Covariate Evaluation

Several potential pre-treatment covariates were evaluated for CUPED implementation.

| Covariate | Correlation with Review Score |
|------------|------------:|
| Delivery Days | -0.221 |
| Total Payment Value | -0.077 |
| Freight Value | -0.046 |
| Product Weight | -0.017 |

Although delivery_days produced meaningful variance reduction, it is not a true pre-treatment covariate because it occurs after treatment exposure and therefore partially captures the treatment effect itself.

Alternative pre-treatment variables available in the dataset exhibit only weak correlations with review_score and therefore provide limited variance-reduction potential.

This finding highlights an important practical limitation of CUPED: the technique is most effective when strong pre-experiment predictors of the outcome metric are available.

Despite this limitation, the exercise demonstrates the mechanics of CUPED and the relationship between covariate strength and variance reduction.